In [1]:
import re
import pandas as pd
import requests
import yaml
import time
import random
import os

In [2]:
# Load credentials from creds.yaml
with open("creds.yaml", "r") as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as e:
        print(f"Error loading YAML file: {e}")

# initialize SocialCrawl API key to use for requests
socialcrawl_key = creds["socialcrawl"]["api_key"]

# Used to test the SocialCrawl API for Instagram profile data
res = requests.get(
    "https://www.socialcrawl.dev/v1/instagram/profile",
    params={"handle": "instagram"},
    headers={"x-api-key": socialcrawl_key},
    timeout=15,
)

# Make sure the request was successful
print(res.status_code)
print(res.text[:500])

200
{"success":true,"platform":"instagram","endpoint":"/v1/instagram/profile","data":{"author":{"id":"25025320","username":"instagram","display_name":"Instagram","avatar_url":"https://scontent-bru2-1.cdninstagram.com/v/t51.82787-19/550891366_18667771684001321_1383210656577177067_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=scontent-bru2-1.cdninstagram.com&_nc_cat=1&_nc_oc=Q6cZ2gEBOdBXRM0BSpdCsUteBU43AITUe4t0lTFn7kW-CbhwMsXHRkDvEGTu9HjmLnnoakE&_nc_ohc=mQBUJxXOw3sQ7kNvw


In [3]:
# Downloading recent posts (SocialCrawl gives 100 credits TOTAL)
def get_posts(handle, max_pages=3):
    rows, cursor = [], None
    for _ in range(max_pages):
        params = {"handle": handle}
        if cursor:
            params["next_max_id"] = cursor
        res = requests.get(
            "https://www.socialcrawl.dev/v1/instagram/profile/posts",
            params=params, headers={"x-api-key": socialcrawl_key}, timeout=30,
        )
        res.raise_for_status()
        body = res.json()
        data = body["data"]
        if len(rows) == 0:
            print(body.get("credits_remaining"))
            print(str(data)[:1500])
        items = data if isinstance(data, list) else (data.get("posts") or data.get("items") or [])
        rows.extend(items)
        cursor = (data.get("next_max_id") if isinstance(data, dict) else None) or body.get("next_max_id")
        if not cursor:
            break
        time.sleep(random.uniform(1, 3))
    return rows

# getting posts from Instagram account
# diu.cpc (computer programming club)
# diu_robotics_club (robotics club)
# CHANGE THIS TO THE ACCOUNT YOU WANT TO SCRAPE
raw = get_posts("diu.cpc")


91
{'items': [{'post': {'id': '4034022886079642042', 'url': 'https://www.instagram.com/p/Df7unEUkrm6/', 'content': {'text': '𝗗𝗜𝗨 𝗧𝗮𝗸𝗲-𝗢𝗳𝗳 𝗣𝗿𝗼𝗴𝗿𝗮𝗺𝗺𝗶𝗻𝗴 𝗖𝗼𝗻𝘁𝗲𝘀𝘁, Fall 𝟮𝟬𝟮𝟲\n\nCompetitive programming is widely recognized as an effective way to enhance problem-solving abilities and strengthen programming skills. To encourage students’ interest in programming and foster a competitive programming mindset among them, DIU CPC from the Department of Computer Science and Engineering, DIU, is once again organizing the Take-Off Programming Contest, Fall 2026. This contest provides an excellent platform for participants to evaluate their current skills and gain valuable experience in a competitive environment.\n\nOnly students from the 1st semester and 2nd semester are eligible to participate. This is an individual contest, so team participation is not allowed. All participants will receive food and a T-shirt.\n\n🔗 Registration Link:  cutt.ly/7yAWDRam\n💰 Registration Fee: 500 BDT\n⏳ Registration Dea

In [4]:
# Dictionaries: words/phrases that signal each concept (classifier)
ai_terms = [
    "ai", "a.i.", "artificial intelligence", "machine learning",
    "generative ai", "genai", "chatgpt", "gemini", "claude", "llm",
    "deep learning", "neural network", "data science", "prompt engineering",
]
event_terms = [
    "event", "seminar", "workshop", "conference", "symposium", "webinar",
    "talk", "meetup", "bootcamp", "hackathon", "summit", "masterclass",
    "competition", "register", "join us",
]

def make_pattern(terms):
    """Match whole words only, so 'ai' does NOT match 'said', 'maintain', 'email'."""
    return re.compile(r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b", re.IGNORECASE)

ai_pattern = make_pattern(ai_terms)
event_pattern = make_pattern(event_terms)

def classify_post(caption):
    text = re.sub(r"\s+", " ", str(caption or ""))
    ai_hits = sorted({m.lower() for m in ai_pattern.findall(text)})
    event_hits = sorted({m.lower() for m in event_pattern.findall(text)})
    return {
        "club": "diu.cpc", # CHANGE THIS TO THE ACCOUNT YOU WANT TO SCRAPE
        "is_ai": len(ai_hits) > 0,
        "is_event": len(event_hits) > 0,
        "ai_event": len(ai_hits) > 0 and len(event_hits) > 0,
        "ai_matches": ai_hits,
        "event_matches": event_hits,
    }

In [5]:
def find_caption(item):
    post = item.get("post", item)
    content = post.get("content") or {}
    return content.get("text") or ""

posts = pd.DataFrame({
    "url":     [i.get("post", i).get("url") for i in raw],
    "caption": [find_caption(i) for i in raw],
})
posts = pd.concat([posts, posts["caption"].apply(classify_post).apply(pd.Series)], axis=1)

print(len(posts), "posts |",
      (posts.caption != "").sum(), "with captions |",
      posts.is_ai.sum(), "AI |",
      posts.is_event.sum(), "event |",
      posts.ai_event.sum(), "AI event")

OUT = "Data/diu_ai_promotion_classified.csv"

#CHANGE THIS TO THE ACCOUNT YOU'RE SCRAPING
posts["club"] = "diu.cpc"

# doing this to avoid duplicates if the script is run multiple times
if os.path.exists(OUT):
    posts = posts[~posts.url.isin(pd.read_csv(OUT).url)]

# appending new posts to the csv
posts.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
print(len(posts), "new posts added")
posts[posts.ai_event]

12 posts | 12 with captions | 9 AI | 8 event | 7 AI event
1 new posts added


,url,caption,club,is_ai,is_event,ai_event,ai_matches,event_matches
